# Implementing a simple Retrieval-augmented generation (RAG)
This Lab is based on a workshop created by our friends of the [Swiss-AI-Center](https://www.hes-so.ch/swiss-ai-center).

Original Authors:
- Jonathan Guerne, Research assistant at Haute Ecole Arc Ingénierie, Switzerland
- Henrique Marques Reis, Research assistant at Haute Ecole Arc Ingénierie, Switzerland
- Célien Donzé, Scientific Collaborator at HEIA-FR, Switzerland

Adapted by:
- Francesco Carrino, Professor at Haute Ecole Ingénierie, Valais/Wallis Switzerland

## Introduction
This notebook explains how to create a RAG (Retrieval Augmented Generation) system to answer questions about online or PDF documents. We will use a self-hosted LLM with Ollama to generate answers to the questions. Additionally, we will use a vector database to retrieve relevant documents for answering the questions.

### Documentation and references
Technologies and tools used in this lab:
- [Large Language Model (LLM)](https://en.wikipedia.org/wiki/Large_language_model), a model that generates text (typically) baseded on the transformer architecture. In this lab, we will used opensource models like LLAMA 3.x.
- [Ollama](https://ollama.com/): Ollama is a platform that democratizes access to LLMs by enabling users to run them locally on their machines. In this lab, it will be used to run the LLMs on your machine or in the cloud (Kaggle).
- [Vector database](https://en.wikipedia.org/wiki/Vector_database): A Vector Database is a relational database system specifically designed to process vectorized data. Unlike conventional databases that contain information in tables, rows, and columns, vector databases work with vectors–arrays of numerical values that signify points in multidimensional space. When used with text data, vector databases can store and retrieve information based on the semantic meaning of the text, rather than just the text itself. Through a process called embedding, text data is converted into vectors that represent the **meaning** of the text. This meanse text with similar meaning is stored close to each other in the vector space. We will use [FAISS](https://python.langchain.com/docs/integrations/vectorstores/faiss/) an opensource library for efficient similarity search and clustering of dense vectors. In this lab, it will be used to store information about pdf and web documents.
- [LangChain](https://python.langchain.com/docs/introduction/): LangChain is a framework for developing applications powered by large language models (LLMs).  It is based on the idea of "chains", where a chain defines sequences of actions, where each step can involve querying an LLM, prompt management, manipulating data, or interacting with external tools applications. In this lab, it will be used to put together the LLM and the vector database to create a RAG system.
- [Retrieval-augmented generation (RAG)](https://en.wikipedia.org/wiki/Retrieval-augmented_generation): RAG is a technique to improve LLMs by adding a retrieval component from an external data source. This is the main goal of this tutorial.

### Structure and goals
This lab is divided into 4 parts. The first 3 parts, you will be guided to install and implement a simple RAG working on pdf files. Your goal in this part understand the code. We provide some questions to guide your learning through the code.

The last part is a challenge where you will have to use what you learnt to implement a RAG working on web pages.

1. **Installation**: Install the necessary libraries and download the data. We provide two guides: one for running the code on Kaggle (or Google Colab), and another for running the code locally.
2. **Creating and manipulating prompts**: We will create prompts and create simple query the LLM and the vector database.
3. **Creating a RAG system**: We will create a simple RAG system to answer questions about pdf documents. The pdf will be embedded in the vector database and the LLM will generate the answers.
4. **Challenge**: In this last part, *you* will implement a RAG system to answer questions about web pages.

---

## 1. Installation

### 1.0 Selecting the model



If your machine is not very performant, you can use Kaggle or Google Colab to run this notebook. The following code block installs the necessary packages.
If you choose to run the code on Kaggle, login to your account and import this notebook. It is raccomanded to have a verified account to have access to the GPU.

As model for this lab we propose llama3.1:8b. Thi model is a (*relatively*) small version of LLAMA 3.1 (it requires about 5GB on your hard disk), making it available for use on your notebook machine environment. We selected this model because allows "tool calling" a functionality useful for the second activity of this lab.

If you want to try other models, please refer to https://ollama.com/search for checking the updated list of models available on the Ollama model repository.

In [1]:
model_name = "llama3.1:8b"  # or 'llama3.2:1b'

Below you can find the instraction to install the necessary packages either on Kaggle (Section 1.1) or on your local machine (Section 1.2).

### 1.1 Installing Ollama (Kaggle or Google Colab)

If you are in Kaggle or Google Colab, you can install Ollama using the following code block. If want to run this notebook locally, see the next section.

These instructions are based on the Medium post titled [How to Run Ollama Models on Google Colab and Kaggle: A Complete Guide](https://medium.com/@mradul18varshney/how-to-run-ollama-generative-ai-models-on-google-colab-and-kaggle-a-complete-guide-3e01fc12512f).

In [ ]:
# Package installation (ONLY on Kaggle or Google Colab)
!pip install langchain langchain-classic langchain-huggingface langchain-community faiss-cpu pymupdf pypdf sentence_transformers rich wget python-dotenv cryptography unstructured accelerate

NOTE: you may receive an error message such as `ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.`. This is a known issue and can be ignored.

#### 1.1.1 Setting up the environment (Kaggle or Google Colab)

This update package list and install curl, which then use to fetch Ollama installation script.

In [ ]:
# Update system files and install curl and zstd
!apt-get update && apt-get install -y curl && apt-get install -y zstd

#### 1.1.2 Installing Ollama (Kaggle or Google Colab)

This will install Ollama automatically.

You will probably get some warning messages such as:

- *WARNING: systemd is not running*
- *WARNING: Unable to detect NVIDIA/AMD GPU. Install lspci or lshw to automatically detect and install GPU depend*

You can ignore these messages.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

#### 1.1.3 Starting the Ollama server (Kaggle or Google Colab)

Here, we use Python’s `subprocess.Popen()` to run the Ollama server in the background, listening on the specified host and port (127.0.0.1:11438). The server will be available for communication via the API.

You will need this address later for the RAG model.

In [ ]:
import subprocess
import time
import os

# Set the environment variable for the Ollama host and port
os.environ['OLLAMA_HOST'] = '127.0.0.1:11438'

# Start the Ollama server in a subprocess
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for the server to start
print("Starting Ollama server...")
time.sleep(1)  # Wait for the server to initialize

print("Ollama server is running on 127.0.0.1:11438")

#### 1.1.4 Pulling an open-source LLM model (Kaggle or Google Colab)
This command downloads the llama3.1:8b (selected above) model from Ollama’s servers making it available for use on your notebook machine environment.

Also, refer here https://ollama.com/search for checking the updated list of models available on the Ollama model repository.

In [ ]:
# Pull a model (e.g., llama)
print("Pulling the llama model...")
subprocess.run(['ollama', 'pull', model_name]) # model_name is defined at the beginning of this code block

#### 1.1.4 Interacting directly with the model (Kaggle or Google Colab)

To test the installation and the model, we can use the following code block to interact with the model via HTTP requests.

In [ ]:
import requests
import json

def generate_response(query):
    # Define the API URL for the same host as mentioned when starting the server.
    url = "http://localhost:11438/api/chat"
    
    # Define the payload (data) to send in the POST request
    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ],
        "keep_alive": 0,
        "stream": False
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    # Make the POST request to the API
    try:
        response = requests.post(url, data=json.dumps(payload), headers=headers)
    
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            output = response.text
            return output
        else:
            print(f"Failed to get a response. Status code: {response.status_code}")
            return ("Error response:", response.text)
    except Exception as e:
        print(f"An error occurred: {e}")

Once the server is running and the model is pulled, you can start interacting with the model using Python code. In this example, the `generate_response()` function sends a POST request to the Ollama API, passing in the user query. The model processes the query and returns a generated response.

*Note: If you change the model you will get the error that the model is not downloaded and you have to pull it using ollama pull. Also, check the model name to match with the model you pulled using `ollama pull` command.*

In [ ]:
# Using the function to get the output
# The output is in JSON format, and you can parse it using Python’s json module.

query = "What is the difference between AI and ML? Give a short answer."
output = generate_response(query)
json.loads(output)['message']['content']

#### 1.1.5 Using Ollama’s Python SDK (Kaggle or Google Colab)

To interact with the Ollama server, you can use the Python SDK. The SDK is available on PyPI and can be installed using pip. The Ollama python SDK is a wrapper around the Ollama API, which allows you to interact with the server using Python code. The SDK provides a simple and easy-to-use interface for sending requests to the server and receiving responses from the model. This is what we will use in this lab to interact with the model.

In [ ]:
!pip install ollama langchain_ollama

The code below interacts with the Ollama model using the Python SDK to send a query and print the generated response.

*Note: (again) If you change the model you will get the error that the model is not downloaded and you have to pull it using ollama pull.*

In [ ]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model=model_name, messages=[
  {
    'role': 'user',
    'content': 'Show me the lyrics of any song?',
  },
])
print(response['message']['content'])

If all works, you should see the generated response from the model. In my test, I got the lyrics from the song "Yesterday" by The Beatles. What did you get?

- - - 

### 1.2 Installing Ollama (locally on your machine)

This guide will help you to install Ollama on your local machine. You need a fairly good machine to run LLMs models and space in your hard disk to store the LLM models (several GB).  If you have a machine with a GPU, you can run the models faster.
However, there are lighter models that can run on CPU and provide results good enough for this lab and many applications (e.g., you can try "llama3.2:1b", or "qwen2.5:0.5b" model).

#### 1.2.1 Download and install Ollama
Ollma is available for macOS, Linux, and Windows. Go to the official [website](https://ollama.com/) and download the version for your operating system. Follow the instructions to install it.

#### 1.2.2 Running Ollama
After installing Ollama, you should see the Ollama icon in your system tray. You can use it to see the logs and quit the application.

Usuall, after the installation Ollama is *already* running. 

1. [Optional] If you want to run it again, you can run the following command in your terminal:

    ```bash
    ollama serve
    ```

2. To pull a model, go to the [Ollama model list](https://ollama.com/search) and select a model suitable for your machine. Start humble with a small model. You can for instance try the small version of [LLAMA 3.2](https://ollama.com/library/llama3.2).
Follow the instractions for the model you selected. For example, we suggest you try LLAMA 3.1 (the 8B parameters version) or LLAMA 3.2 (the 1B parameters version). To use the suggested LLAMA 3.1 version, run in your terminal:

    ```
    ollama run llama3.1:8b
    ```
    The `run` command will pull (download) the model to your machine and then run it, exposing it via the API started with `ollama serve`.

3. [Optional] If you want to *pull* a differen model, you can run:

    ```bash
    ollama pull <model_name>
    ```
    The commanda `ollama list` will show you the list of models available on your machine. To switch between models available on your machine, you can just use the `run` command. The `stop` command will stop a running model.

NOTES:
- If the model is not present in your machine, the `run` command will autamtically try to pull it
- Models takes a lot of space in your hard disk. Make sure you have enough space before pulling a model. Remove unnecessary models using the `ollama remove <model_name>` command.

#### 1.2.3 Get Ollama address
In order to interact with the model, you need to know the address where the model is running. Ollama binds to the local address 127.0.0.1 on port 11434 by default.

This means that you can note http://localhost:11434

#### 1.2.4 Package installation (with uv)
We will install all the necessary packages (ollama-related packages included) using uv. We provide a `pyproject.toml` file with the necessary dependencies. We assume you have [uv installed](https://github.com/hei-synd-aml/lab-0-TutoUv) on your machine. Open a terminal in the project location and run the following command to install the packages.

```bash
uv sync
```

---

## 2. Creating a prompt and interacting with the model

OK, now that you have the Ollama server running (either on Kaggle or on your Local machine) and you pulled the model, let's start by creating a prompt and interacting with the LLM model.
In this section, we will create a prompt and interact with the model to generate text.

### Imports

In [2]:
import os
from pathlib import Path

import wget
#from dotenv import load_dotenv
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.prompts import PromptTemplate
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaLLM
from rich.console import Console
from rich.markdown import Markdown

console = Console()

### Constants

A little bit of configuration depending on your system (Kaggle or local). Select the right values for your case.

In [3]:
# OLLAMA_ADDRESS = "http://localhost:11438" # use this if you are running Ollama on Kaggle or Colab
OLLAMA_ADDRESS = "http://localhost:11434"  # use this if you are running Ollama on your local machine

model_name = model_name # Defined  replace with the LLM name you pulled from the OLLAMA


### Connecting to LLM

NOTE:
- The first time you run this cell, it may take a while to connect to the model, as the model may need to be loaded into memory.
- if you get "ConnectionError", check if the Ollama server is running and the address is correct.

In [5]:
llm = OllamaLLM(
    model=model_name,
    base_url=OLLAMA_ADDRESS,
    temperature=0.6,
    stop=["<end_of_turn>"],
)

llm.generate(["Hello, tell me a joke about infotronics."]).generations[0][0].text

'A bit of a niche topic! Here\'s one:\n\nWhy did the infotronic system go to therapy?\n\nBecause it had a lot of "byte-sized" problems and was struggling to "process" its emotions!\n\nHope that one charged up a smile on your face!'

**Question**:
- What is the role of the temperature parameter in a LLM? Try to ask the same question with very low values of temperature (0.0001) or high values of temperature. Check the code of the model you selected to see the range of temperature. For llama 3.1:8b the suggested temperature=0.6

**Answer**:

- The temperature parameter in a language model (LLM) controls the randomness of the model's output.
    - A lower temperature value (closer to 0) makes the model more deterministic, favoring higher probability words and resulting in more predictable and repetitive text.
    - A higher temperature value (closer to 1) increases randomness, allowing for more creative and diverse responses by giving less probable words a better chance of being chosen.
    - Adjusting the temperature helps balance between coherence and creativity in the generated text.

### Creating a prompt

A prompt is generally divided into two parts: the context and the question.

The context provides the information that the model will use to generate its answer, while the question specifies what the model is expected to respond to.
Typically, the context contains a system prompt, which explains the expected behavior from the model, and other additional information that can help the model to provide the right answer. Typically, when used for chatbots, in the context there is a summary (or the entierity!) of all the previous exchange the user had with the model in that conversation. 

LangChain uses templates and markers indicating where to insert the user's question and the context within the template. For more infomation, you can check [Langchain prompt templates documentation](https://python.langchain.com/v0.2/docs/concepts/#prompt-templates).

In [6]:
# A langchain's template is nothing more than a string with {markers} indicating placeholder where other text will be inserted when creating a prompt.

template = """
You are an helpful assistant that answer the question in detail.

Human input: {question}
Assistant:"""

prompt = PromptTemplate(input_variables=["question"], template=template)
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are an helpful assistant that answer the question in detail.\n\nHuman input: {question}\nAssistant:')

### Creating the chain and start a conversation

In [7]:
# Simple chain putting together a prompt and a model 
simple_llm_chain = prompt | llm

In [10]:
result = simple_llm_chain.invoke(input="What does the term RAG refer to in machine learning?") # we ask a question to the LLM

console.print(Markdown(result)) # we print the output proposed by the LLM

In machine learning, RAG stands for "Reparameterization Autoencoders with Global Attention" or "Recurrent Attention
Generators" (there are actually two related concepts with this acronym, but I'll cover both).                      

Reparameterization Autoencoders with Global Attention (RAG): RAG is a type of neural architecture designed for     
sequence-to-sequence tasks, such as machine translation, text summarization, and question answering. It combines   
the benefits of autoencoders and attention mechanisms.                                                             

In traditional autoencoders, the input is encoded into a lower-dimensional representation, and then the encoded    
representation is decoded back to the original input. However, this process can be brittle and sensitive to changes
in the input.                                                                                                      

RAG addresses this issue by using a reparameterization trick, which involves introducing noise to the encoded      
representation during training. This allows the model to learn a more robust and flexible representation of the    
input sequence.                                                                                                    

The attention mechanism is used to focus on specific parts of the input sequence when generating the output. This  
is achieved by computing a weighted sum of the input sequence, where the weights are learned during training.      

Recurrent Attention Generators (RAG): RAG is also used to refer to a type of generative model, specifically a      
variant of the Generative Adversarial Network (GAN) architecture.                                                  

In this context, RAG uses a recurrent neural network (RNN) to generate sequences, such as text or images. The      
attention mechanism is used to focus on specific parts of the input sequence when generating the output.           

RAG is designed to improve the quality and diversity of generated samples by using a combination of attention and  
reparameterization techniques.                                                                                     

Key benefits of RAG: The RAG architecture has several benefits, including:                                         

 1 Improved robustness: By using a reparameterization trick, RAG models are more robust to changes in the input    
   data.                                                                                                           
 2 Increased flexibility: The attention mechanism allows RAG models to focus on specific parts of the input        
   sequence, improving their ability to generate high-quality output.                                              
 3 Better handling of long-range dependencies: RAG models can effectively handle long-range dependencies in the    
   input sequence, which is particularly useful for sequence-to-sequence tasks.                                    

Use cases: RAG models have been applied to a variety of tasks, including:                                          

 1 Machine translation: RAG models have been used to improve the accuracy and fluency of machine translation       
   systems.                                                                                                        
 2 Text summarization: RAG models have been used to generate concise and accurate summaries of long documents.     
 3 Question answering: RAG models have been used to improve the accuracy and relevance of question answering       
   systems.                                                                                                        

I hope this detailed explanation helps you understand the concept of RAG in machine learning!

**Questions**:
- What is the relationship between the `input` parameter in `llm_chain.invoke(input="")` and the prompte template defined above?
- Did the model generate a good answer? Why?
- Feel free to change the prompt and the input and see what happens. Try to consider this point: how can you judge the quality of the answer? How could you create a proper evaluation protocol to evaluate the quality of the answers? How is it done in the industry/research field?

**Answers**:
- The `input` parameter is used to pass the question to the model. The question is inserted using the `{question}` marker.
NOTE: different LLMs may have different markers for the question and context. Check the documentation of the model you are using. This is a true challenge of prompt-engineering.
- The model generated a bad answer. The model does not have access to the internet and it was trained on data before the conference mentioned in the question. It is very improbable that it gives a correct answer.
- Evaluating a LLM is not easy. There are several ways to evaluate a LLM. The most common way is to use human evaluation, where humans evaluate the quality of the answers. This is very expensive, time-consuming, and subjective. Another way is to use automatic evaluation metrics. These metrics are based on the similarity between the generated text and the reference text (e.g., a collection of answers proposed by experts to the same questions).

---

## 3. Using the RAG

For the moment, we have a model that can generate answers to questions. To provide answers, it only uses its knolwege got at training time and the context we provide. But what if we could provide the model with a set of documents and ask it to retrieve the most relevant ones to answer the question?
That is the idea behind the Retrieval-Augmented Generation (i.e., literally, we augment the generation of text by enriching the context with information contained in relevant documents).

A typical use case is a company having a huge internal knowledge base (e.g., a collection of documents, web pages, etc.) and wanting to provide its employees with a chatbot that can answer questions about the knowledge base. The chatbot will retrieve the most relevant documents from the knowledge base and use them to generate the answer. This is particular interesting for companies that have a lot of internal documents that may contain sensitve data.

<img src="./img/RAG_schema.png" alt="Generic schema of a RAG system" width="600"/>

[*(Source of the image)*](https://www.promptingguide.ai/research/rag)

### Exploring pdf files

Here we demonstrate RAG using a collection of *pdf* documents. We will use the scientific articles available online in open access. We will download the documents and store them in the `data` folder. You are free to test the code with your own documents. Just make sure they are in the `data` folder.

NOTE: *pdf* documents are surprisingly complex documents. They can contain text, images, tables, etc. Actually, two different *pdf* documents can have very different structures. Some additional preprocessing may be needed to extract the text from the pdfs. However, for the purpose of this lab, the code provided below should work for most of the documents.

In [12]:
# Create the "data/PDFs" folder if it doesn't exist
PDF_FOLDER = Path("data/PDFs")
os.makedirs(PDF_FOLDER, exist_ok=True)

urls = [
    "https://simg.baai.ac.cn/paperfile/25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf", # Title: "Retrieval-Augmented Generation for Large Language Models: A Survey"
    "https://arxiv.org/pdf/2405.07437.pdf", # Title: "Evaluation of Retrieval-Augmented Generation: A Survey"

]

# Download the PDFs
for url in urls:
    name = url.split("/")[-1]
    if not (PDF_FOLDER / name).is_file():
        filename = wget.download(url, f"data/PDFs/{name}")
console.print("Pdf file downloaded successfully.", style="bold green")

Pdf file downloaded successfully.

### Loading and preprocessing PDF files

In [13]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', r'(?<=\. )', r'(?<=\, )', ' ', ''] # the r indicates a raw string
)

***Questions***
- Investigate and explain the role of `chunk_size` and `chunk_overlap` in the code above.
- What are the advantages and disadvantages of having bigger/smaller `chunk_size`?
- What are the advantages and disadvantages of having bigger/smaller `chunk_overlap`?
- What is a difference between a token and a chunk?
- What is the parameter `separators` used for?
- What does the **Recursive** Character Text Splitter does differently compared to a **Simple** character text splitter? 

***Answers***
- The `chunk_size` parameter defines the size of the chunks in which the text is divided. A larger chunk size will result in fewer chunks, while a smaller chunk size will result in more chunks. The `chunk_overlap` parameter defines the overlap between chunks. A larger overlap will result in more redundancy between chunks, while a smaller overlap will result in less redundancy.
- Advantages of having a bigger `chunk_size` include faster processing and reduced memory requirements, while disadvantages include reduced granularity and less accurate results. Advantages of having a smaller `chunk_size` include increased granularity and more accurate results, while disadvantages include slower processing and increased memory requirements.
- Advantages of having a bigger `chunk_overlap` include increased redundancy and improved performance, while disadvantages include increased memory requirements and slower processing. Advantages of having a smaller `chunk_overlap` include reduced redundancy and improved accuracy, while disadvantages include reduced performance and increased memory requirements.
- A token is the smallest unit of text that the model can process. A chunk is a larger unit of text that is used to divide the text into manageable pieces. Chunks are used to reduce the memory requirements of the model and improve performance.
- The `separators` parameter is used to define the characters that separate chunks. By default, the separator is set to `["\n\n", "\n", " ", ""]`, which means that chunks are separated by two newline characters. This allows the model to identify the boundaries between chunks and process them separately.$
- The **Recursive** Character Text Splitter tries to split the text recursively into smaller chunks based on separators and until the chunk size is reached. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text ([source](https://python.langchain.com/docs/how_to/recursive_text_splitter/)).

In [14]:
# All data will be in the list documents
documents=[]

In [15]:
from langchain_classic.document_loaders import DirectoryLoader, PyPDFLoader
# Load and process the text files
# loader = TextLoader('single_text_file.txt')
loader = DirectoryLoader(PDF_FOLDER, glob="./*.pdf", loader_cls=PyPDFLoader)

pdf_docs = loader.load()
len(pdf_docs)

# tokenize pdf
documents.extend(text_splitter.split_documents(pdf_docs))
len(documents)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong po

562

#### What is a "Document" in langchain?
In LangChain, a  [document](https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html) is composed of two elements: the text (contained in the field `page_content`) and the metadata (contained in the field `metadata`).

The metadata is a dictionary that can contain any information you want to store about the document. In this case, we store the URL of the website.

In [16]:
# To understand what is going on, we print a document. The text:
print(documents[1].page_content)

Abstract. Retrieval-Augmented Generation (RAG) has recently gained traction
in natural language processing. Numerous studies and real-world applications
are leveraging its ability to enhance generative models through external informa-
tion retrieval. Evaluating these RAG systems, however, poses unique challenges
due to their hybrid structure and reliance on dynamic knowledge sources. To
better understand these challenges, we conduct A Unified Evaluation Process of


In [17]:
# To understand what is going on, we print a document. The text:
print(documents[1].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-07-04T00:23:56+00:00', 'author': '', 'keywords': '', 'moddate': '2024-07-04T00:23:56+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\PDFs\\2405.07437.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}


**Question**:
- why is important to also store metadata when doing RAG? Any idea?

**Answer**:
- Metadata are very important. They usually contains information useful to link a chunk of text, to its source. Therefore, metadata allow being sure that a model is not hallucinating and navigate back to the source of the information.

### Embedding a PDF in a vectorstore

The code below creates a vectorstore from the documents. The vectorstore is a database that stores the embeddings of the documents. The embeddings are used to retrieve the most relevant documents for a given question.
It may take a while to create the vectorstore, depending on the number of documents and the size of the documents.

However, if you have new documents, you can just add them to the vectorstore using the `add_documents` method instead of recreating the vectorstore from scratch. 

In [18]:
VECTORSTORES_DIR = Path("data/vectorstores_pdf")

In [23]:
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5" # locally you can also use smaller models. Check the HuggingFace model hub for more models with different sizes, performance and languages: https://huggingface.co/spaces/mteb/leaderboard

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    # to use the "cuda" configuration, you need an nvidia GPU, and to install 
    # On Kaggle, you set to use as accelerator GPU P100 (you need a verified account)
    # model_kwargs={"device": "cpu"}, # change to cuda -> cpu if you do not have a Nvdia GPU
    model_kwargs={"device": "cpu"}, # change to mps for MacOS M1/M2
    encode_kwargs={"normalize_embeddings": True},
)

The instructions below embeds the documents in the vectorstore. Then, we can save the vectorstore to disk. In this way, we can load the vectorstore and the documents at a later time without having to re-embed the documents. If we get new relevant documents, we can add them to the vectorstore and re-embed them. If interested, you can check the [FAISS documentation](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html) for more information.

In [24]:
vectorstore = FAISS.from_documents(documents=documents, embedding=embedding_model)

In [25]:
vectorstore.save_local(VECTORSTORES_DIR)

### Loading a vectorstore
We already have the vectorstore with the pdfs embedded in the variabale `vectorstore`. For completeness, in the code above, we show how to load the vectorstore from disk.

In [26]:
vectorstore = FAISS.load_local(
    VECTORSTORES_DIR, embedding_model, allow_dangerous_deserialization=True
)

### New prompt

In RAG we need to add another marker to indicate where the new information (or context) should be inserted for this we use the variable named `{context}`.

In [27]:
prompt = """
Use the following pieces of context to answer the question at the end.
Don't try to make up an answer and only use the information you know.
Use three sentences maximum and keep the answer as concise as possible.
You must answer in english. If you don't know the answer, say "I don't know".
Context:
{context}

Question:
{input}

Answer:
"""

prompt_template = PromptTemplate(input_variables=["context", "input"], template=prompt)
prompt_template

PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nUse the following pieces of context to answer the question at the end.\nDon\'t try to make up an answer and only use the information you know.\nUse three sentences maximum and keep the answer as concise as possible.\nYou must answer in english. If you don\'t know the answer, say "I don\'t know".\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n')

***Questions***
- If you look closely, you will notice that the prompt above has 4 regions into which we can insert (or not) information (the system prompt, the context, the input and the answer). What are these regions? Explain the role of each one.

***Answers***
- System prompt: This is the prompt that will be sent to the model. It contains the context and the question.
- Context: This is the information that the model will use to generate its answer. It can be a document, a paragraph, or any other relevant information.
- Question: This is the question that the model is expected to respond to.
- Answer: This is where the model will insert its answer. This is the answer showed to the user.

**Tips and Tricks**: You can modify the system prompt to get an answer closer to the one you are looking for.

- Change the language, style, or length.
- Add more information to the prompt, such as suggesting a style or tone for the answer (e.g., "be concise", "be formal", "be informal", "be friendly", etc.).
- Provide more context (e.g., "the text is a scientific paper", "the text is a news article", etc.).
- Specify the question type (e.g., "the question is about the content of the text", "the question is about the author of the text", etc.).
- Define the answer format (e.g., "the answer should be a summary of the text", "the answer should be a list of keywords", etc.).
- Include information about the user (e.g., "the user is a scientist", "the user is a student", etc.).
- Mention details about the model (e.g., "the model is a chatbot", "the model is a search engine", etc.).
- Indicate the desired output format (e.g., "the output should be in JSON format", "the output should be in XML format", etc.).
- Provide specific examples to illustrate the impact of prompt modifications.
- Highlight any limitations or constraints of the model.
- And more...

**Notes and Warnings**:

- Different models may have different markers for the question and context. Check the documentation of the model you are using. This is a true challenge of prompt-engineering.
- Some models (especially smaller ones) may not support different languages.

### Similarity search

<img src=".\img\Similarity_search_schema.png" alt="Generic schema of a RAG system" width="600"/>

[*(Source of the image)*](https://python.langchain.com/docs/concepts/vectorstores/)

In [28]:
# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="mmr", #  Can be "similarity" (default), "mmr", or "similarity_score_threshold".
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In the code above, multiple things go on at the same time.
[create_stuff_documents_chain](https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain/) combines documents by concatenating them into a single context window. 

**Questions**:
- What is a retriever? 
- What is the meaning of the parameter: "mmr"? What is the difference between "mmr", "similarity" and "similarity_score_threshold"?
- What is the meaning of the parameter "k" and how increasing/decreasing it may impact the results?
- When using "mmr", there are other parameters for the retriever. One of them is `lambda_mult`. What is the meaning of this parameter and what are the possible values? 
- What `create_retrieval_chain` does?

**Answers**:
- A retriever is a component that retrieves relevant documents from a database based on a query.
- `"mmr"`, `"similarity"` (default), or `"similarity_score_threshold"` define the three search strategies implemented in FAISS.
    - `mmr` stands for Maximal Marginal Relevance. It is a method used to diversify the results of a search query by selecting documents that are both relevant and dissimilar to each other. The "lambda_mult" parameter specifies the degree of diversity among the results (see below).
    - The `similarity` search strategy retrieves documents based on their similarity to the query.
    - The `similarity_score_threshold` also search strategy retrieves documents based on their similarity to the query. However, it allows defining a `score_threshold` parameter, which specifies the minimum similarity score required for a document to be retrieved.
- The parameter `k` specifies the number of documents to retrieve. Increasing `k` will result in more documents being retrieved, while decreasing `k` will result in fewer documents being retrieved. The value of `k` should be chosen based on the desired trade-off between accuracy and performance.
- `lambda_mult` (float) – Number between 0 and 1 that determines the degree of diversity among the results with 0 corresponding to maximum diversity and 1 to minimum diversity. Defaults to 0.5.
- `create_retrieval_chain` creates a chain that retrieves relevant documents from a database based on a query. The chain:
    - uses a retriever to retrieve the relevant documents
    - combines the retrieved documents into a single context window
    - passes this information (as part of the context) to the model to generate the answer to the user's question.

### "Chatting" with a pdf

Everything is ready to chat with the model: the content of the pdf is embedded in the vectorstore, the model is running, and the chain is created. Let's ask some question to our pdf! Maybe they can help, where the model alone could not.

In [29]:
result = chain_with_retriever.invoke(input={"input": "What does the term RAG refer to in machine learning?"})

console.print(Markdown(result["answer"]))

The term RAG refers to Retrieval-Augmented Generation, a machine learning approach that combines a pre-trained     
retriever with a pre-trained seq2seq model (generator). It involves retrieving relevant information from a vast    
corpus of documents and using this information to generate responses or text. RAG enhances the quality of          
predictions by allowing developers to attach a knowledge base and avoid retraining the entire large model for each 
specific task.

**Question**:
- What is the size of `result`? What do you expect to find in `result`?
- Let us consider again the "temperature" parameter. For an application that needs to provide accurate information coming from existing document, is it better to use a high or low temperature?
- Suggestion: try to change it and see the difference in the answer.

**Answer**:
- The size of `result` is equal to the parameter `k` discussed above. `result` contains the chunks retrieved by the retriever from the vectorstore. The chunks are ordered by relevance to the query and each chunk has its related metadata. The LLM combines these chunks to generate the final answer (also contained in `result`). To have a better understanding of the `result` variable, just print it!
- The temperature parameter in a language model (LLM) controls the randomness of the model's output. A RAG application, usually, needs to provide accurate information coming from existing documents. In this case, it is better to use a lower temperature value (closer to 0) to make the model more deterministic, favoring higher probability words and resulting in more predictable and repetitive text. This way, the model will be more likely to provide accurate information from the documents.

### Explaining the output

The `result` dictionary contains much more than the simple answer. Thanks to the metadata, we can see the title of the document, the page number, and the context where the answer was found!

In [34]:
console.print(result)

{
    'input': 'What does the term RAG refer to?',
    'context': [
        Document(
            id='6c773a68-5112-4e3f-99ac-a44eba5353ed',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 2,
                'page_label': '3'
            },
            page_content='contents of the survey.\n2 Background\nIn this chapter, we will introduce the definition 
of RAG, as\nwell as the comparison between RAG and other model opti-\nmization techniques, such as 
fine-tuning.\n2.1 Definition\nThe meaning of RAG has expanded in tandem with techno-\nlogical developments. In the 
era of Large Language Mod-\nels, the specific definition of RAG refers to the model, when\nanswering questions or 
generating text, first retrieving rele-'
        ),
        Document(
            id='9fb5b60d-3509-4abc-9582-783fc031c343',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 9,
                'page_label': '10'
            },
            page_content='representations?\nIn RAG, semantic space is the multidimensional space where\nquery and 
Document are mapped. When we perform re-\ntrieval, it is measured within the semantic space. If the se-\nmantic 
expression is not accurate, then its effect on RAG is\nfatal, this section will introduce two methods to help us 
build\na accurate semantic space.\nChunk optimization\nWhen processing external documents, the first step is 
chunk-\ning to obtain fine-grained features. Then the chunks are Em-'
        ),
        Document(
            id='5ed5e89c-2df2-4acb-acc1-9f867e53ce6f',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 9,
                'page_label': '10'
            },
            page_content='Retrieve-Read flow. Self-RAG [Asai et al., 2023b] fol-\nlows the 
decide-retrieve-reflect-read process, introduc-\ning a module for active judgment. This adaptive and\ndiverse 
approach allows for the dynamic organization of\nmodules within the Modular RAG framework.\n4 Retriever\nIn the 
context of RAG, the ”R” stands for retrieval, serving\nthe role in t

**Questions**:
- The `result`dictionary contains thee keys: `input`, `context`, and `answer`. What do they contain? Do they remember you of something?
- Let us focus on the `context` key. It contains a list of documents. Each document is a dictionary with three keys: `id`, `metadata`, and `page_content`. What do they contain? What is the relationship between what you see and the "chunk" you saw before?
- Which of this information is used by the retriever to retrieve the documents?
- Which of this information is used by the LLM to generate the answer?

**Answers**:
- The `result` dictionary contains the input question, the context (the documents retrieved by the retriever), and the answer generated by the model. Do you remember the fields in the `PromptTemplate` above? 
- The `context` key contains a list of documents. Each document is a dictionary with three keys: `id`, `metadata`, and `page_content`.
    - The `id` is the identifier of the document.
    - The `metadata` is a dictionary with the metadata of the document (e.g., the page from which the text was extracted, the title of the document, etc.).
    - The `page_content` is the content of the document (the text). The `page_content` is the chunk of text that was retrieved by the retriever.
- The `page_content` is used by the retriever to retrieve the documents (this is the main source of the embeddings). The `page_content` is also the main text analysed and used by the LLM to generate the answer. However, all the context is passed to the LLM to generate the answer.
- However, we can use the `metadata` to filter the documents and select only the relevant ones. We will show this in the next section.

### Add filters using metadata
In the code above, we did not yet used information in metadata. Here below, weshow how to filter the documents based on just that. For instance, what if we want to limit (filter) the retrieval to document with a specific title?

In [31]:
# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="mmr", #  Can be "similarity" (default), "mmr", or "similarity_score_threshold".
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
        
        # this below is the only new part
        "filter": { 
            "source": "data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf", # Filter by source
        }
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In [32]:
result = chain_with_retriever.invoke(input={"input": "What does the term RAG refer to?"})

console.print(Markdown(result["answer"]))

The term RAG refers to Retrieval-Augmented Generation.

In [33]:
console.print(result)

{
    'input': 'What does the term RAG refer to?',
    'context': [
        Document(
            id='6c773a68-5112-4e3f-99ac-a44eba5353ed',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 2,
                'page_label': '3'
            },
            page_content='contents of the survey.\n2 Background\nIn this chapter, we will introduce the definition 
of RAG, as\nwell as the comparison between RAG and other model opti-\nmization techniques, such as 
fine-tuning.\n2.1 Definition\nThe meaning of RAG has expanded in tandem with techno-\nlogical developments. In the 
era of Large Language Mod-\nels, the specific definition of RAG refers to the model, when\nanswering questions or 
generating text, first retrieving rele-'
        ),
        Document(
            id='9fb5b60d-3509-4abc-9582-783fc031c343',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 9,
                'page_label': '10'
            },
            page_content='representations?\nIn RAG, semantic space is the multidimensional space where\nquery and 
Document are mapped. When we perform re-\ntrieval, it is measured within the semantic space. If the se-\nmantic 
expression is not accurate, then its effect on RAG is\nfatal, this section will introduce two methods to help us 
build\na accurate semantic space.\nChunk optimization\nWhen processing external documents, the first step is 
chunk-\ning to obtain fine-grained features. Then the chunks are Em-'
        ),
        Document(
            id='5ed5e89c-2df2-4acb-acc1-9f867e53ce6f',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 9,
                'page_label': '10'
            },
            page_content='Retrieve-Read flow. Self-RAG [Asai et al., 2023b] fol-\nlows the 
decide-retrieve-reflect-read process, introduc-\ning a module for active judgment. This adaptive and\ndiverse 
approach allows for the dynamic organization of\nmodules within the Modular RAG framework.\n4 Retriever\nIn the 
context of RAG, the ”R” stands for retrieval, serving\nthe role in t

**Questions**:
- Was the answer generated by the model different? 
- Did you notice that the answer was more or less more precise? Why? What are the risks?

**Answers**:
- The answer generated by the model was different. It was only based on the document with the title "session2b".
- The answer may actually be worse. The model is not able to provide a good answer if the documents kept by the filter are not relevant to the question.

---

## 4. Challenge: getting information from a website

Now, it's your turn. Use the code above as inspiration to:
- interact with a website (instead of a PDF)
- create a vectorstore with the information from the website
- save the vectorstore to disk
- add embeddings from another website to the vectorstore (without recreating it from scratch)
- improve the prompt template to get better answers
- chat with the model
- evaluate the output (with and without RAG)

You can choose any website you want. If do not have any inspiration, we suggest to use https://docs.realgames.co/homeio/en/.
You may ask the model to provide you with a list the available detectors in Home I/O. And try to use filters and metadata to improve the results (maybe pointing the actual webpage containing the correct answer...).


Hint: If you need some help to get started check [this](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base).

### Load the website

Loadgin information from a website can be done in multiple ways. In this solution, we show how to use a `WebBaseLoader` (to load a list of pages) and a `RecursiveUrlLoader` to automatically go deeper from a root url.

As for the pdf files, "loading" means extracting text from the website, deviding it in chunks, and appending it to a list of langchain's [documents](https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html).

In [35]:
# All data will be in a new list documents
documents = []

In [36]:
# we define a function to extract the text from the HTML and clean it
import re
from bs4 import BeautifulSoup as Soup

def extract_relevant_text(html: str) -> str:
    soup = Soup(html, "html.parser")
    # Extract content from <p>, <h1>, and <h2> tags
    paragraphs = soup.find_all(['p', 'h1', 'h2'])
    text = "\n".join([para.get_text() for para in paragraphs])
    # Remove unwanted characters like ¶, non-breaking spaces, and newlines
    text = re.sub(r'[\u00b6\u200b\u2002\u2003\u2009\u200c\u200b\u200e\u200f\u2028\u2029]+', ' ', text)
    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [37]:
from langchain_classic.document_loaders import WebBaseLoader # read one or a list of pages
from langchain_classic.document_loaders.recursive_url_loader import RecursiveUrlLoader # read recursively from a root page
from langchain_classic.vectorstores.utils import filter_complex_metadata

is_recurisvely = True

if is_recurisvely:
    # Initialize the RecursiveUrlLoader with the root URL and custom extractor
    url = "https://docs.realgames.co/homeio/en/"
    loader = RecursiveUrlLoader(
        url=url,
        max_depth=3,
        extractor=extract_relevant_text
    )

    # Load the documents
    web_docs_up = loader.load()

    # Initialize the text splitter with desired chunk size and overlap
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

else:
    urls = [
        "https://spl.hevs.io/spl-docs/index.html",
        "https://spl.hevs.io/syrto/AT-Com/atcom-docs/overview.html",
        "https://spl.hevs.io/syrto/AT-Com/atcom-docs/site/index.html"
        ]
    loader = WebBaseLoader(urls)

    web_docs_up = loader.load()

 # Split the loaded documents into smaller chunks
web_docs = text_splitter.split_documents(web_docs_up)

# Filter out metadata that might not be useful for embeddings
documents = filter_complex_metadata(web_docs)

# Output the number of documents after processing
print(f"Number of documents after processing: {len(documents)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Number of documents after processing: 46


As explained in [Section 3](#what-is-a-document?), a document is composed of two elements: the text and the metadata. The metadata is a dictionary that can contain any information you want to store about the document. In this case, we store the URL of the website.

In [38]:
for doc in documents[:5]:
    print(doc.page_content)

About Home I/O Home I/O is an interactive simulation of a smart house and surrounding environment. It was designed to cover a wide range of curriculum targets within Science, Technology, Engineering and Math (STEM). With Home I/O, students will learn about home automation, thermal behavior, energy consumption efficiency and many other subjects that are part of everyday life. The main goal of Home I/O is to introduce home automation concepts using a smart interactive house. Equipped with the most
a smart interactive house. Equipped with the most common automation devices, Home I/O challenges students to design control solutions and understand the impact of its implementation.
Release Notes v1.7.2 (2023/07/11) v1.7.1 (2022/04/05) v1.7.0 (2021/10/27) v1.6.0 (2020/10/07) v1.5.2 (2020/03/19) v1.5.1 (2019/05/02) v1.5.0 (2017/10/16) v1.4.0 (2017/05/01) v1.3.1 (2016/06/01) v1.3.0 (2016/04/07) v1.2.2 (2015/01/06) v1.2.1 (2014/12/19) v1.2.0 (2014/11/19) v1.1.2 (2014/04/01) v1.1.0 (2014/02/28)
Ho

In [51]:
for doc in documents[:]:
    print(doc.metadata)

{'source': 'https://docs.realgames.co/homeio/en/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}
{'source': 'https://docs.realgames.co/homeio/en/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}
{'source': 'https://docs.realgames.co/homeio/en/release-notes/', 'content_type': 'text/html; charset=utf-8', 'title': 'Release Notes - Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}
{'source': 'https://docs.realgames.co/homeio/en/web-server/', 'content_type': 'text/html; charset=utf-8', 'title': 'Web Server - Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}
{'source': 'https://docs.realgames.co/homeio/en/web-server/', 'content_type': 'text/html; charset=utf-8', 'title': 'Web Server - Home I/O', 'description': 'Home I/O Documentation', 'language': 'en'}
{'source': 'https://docs.realgames.co/homeio/en

### Adding the website to the vectorstore

We will use the same code as before to add the website to the vectorstore. Here, we store this information in another vectorstore, so we can keep the information from the pdf and the website separated. In some case, we may want to merge the information from different sources. You can check an example [here](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html).

Here, basically, we used the exact same code as before, but with a different vectorstore.

In [42]:
# we create the vectorstore (it may take a while...)
vectorstore = FAISS.from_documents(documents=documents, embedding=embedding_model)

# if you want to add document to an existing vectorstore, you can use the following code:
# vectorstore.add_documents(documents=documents, embedding=embedding_model)

# Important Note: Make sure that the embedding_model you use for add_documents is the SAME one you used to create the initial vector store. Using a different embedding model will result in inconsistent embeddings within your vector store.

In [43]:
# we save the vectorstore
VECTORSTORES_DIR = Path("data/vectorstores_web")
vectorstore.save_local(VECTORSTORES_DIR)

### Prompting time!

We reuse the same prompt and retriever and chain as before. Therefore, we can reuse the same chain to answer the question.
However, for a sanity check, we try to ask the same question to the model with and without the website information. Let's see if the RAG is actually useful.


In [44]:
prompt = """
Use the following pieces of context to answer the question at the end.
Don't try to make up an answer and only use the information you know.
Consider carefully the context.
Keep the answer as concise as possible.
If you don't know the answer, add after your answer: "I am not sure" and explain why.
Context:
{context}

Question:
{input}

Answer:
"""

prompt_template = PromptTemplate(input_variables=["context", "input"], template=prompt)
prompt_template

PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nUse the following pieces of context to answer the question at the end.\nDon\'t try to make up an answer and only use the information you know.\nConsider carefully the context.\nKeep the answer as concise as possible.\nIf you don\'t know the answer, add after your answer: "I am not sure" and explain why.\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n')

In [45]:
# Simple chain (no context coming from the retrieved documents)
result = simple_llm_chain.invoke(input="List the available detectors in Home I/O.")

console.print(Markdown(result)) # we print the output proposed by the LLM

Home I/O is a home automation system that allows you to monitor and control various aspects of your home's         
security, lighting, temperature, and more. The available detectors in Home I/O include:                            

  1 Motion Detectors: These detectors are placed in strategic locations around the home and can detect movement in 
    a specific area. When motion is detected, the system can send alerts to your smartphone or tablet, and even    
    trigger lights or security cameras to turn on.                                                                 
  2 Smoke and Heat Detectors: These detectors are crucial for home safety and are designed to detect smoke and     
    heat, alerting you and the authorities in case of a fire. They are usually placed in areas where fires are most
    likely to start, such as the kitchen and bedrooms.                                                             
  3 Carbon Monoxide Detectors: Similar to smoke and heat detectors, carbon monoxide detectors are designed to      
    detect the presence of carbon monoxide gas, which can be deadly in high concentrations. They are often placed  
    near fuel-burning appliances like furnaces and water heaters.                                                  
  4 Glass Break Detectors: These detectors are designed to detect the sound of breaking glass, which can indicate a
    forced entry or a window being smashed. They are usually placed near doors and windows.                        
  5 Door and Window Sensors: These sensors are attached to doors and windows and detect when they are opened or    
    closed. They can send alerts to your smartphone or tablet, and can even integrate with your home's security    
    system to trigger alarms and notifications.                                                                    
  6 Water Leak Detectors: These detectors are designed to detect water leaks in areas where they are most likely to
    occur, such as near sinks, toilets, and appliances. They can alert you to potential water damage and help      
    prevent costly repairs.                                                                                        
  7 Temperature Detectors: These detectors can monitor the temperature in specific areas of your home, sending     
    alerts if the temperature exceeds a set threshold. They can be useful for monitoring areas like your home's    
    attic or basement.                                                                                             
  8 Humidity Detectors: These detectors monitor the humidity levels in your home and can alert you to potential    
    issues like mold growth or condensation problems.                                                              
  9 Gas Leak Detectors: These detectors are designed to detect the presence of gases like propane and natural gas, 
    which can be hazardous if they leak.                                                                           
 10 Light Detectors: These detectors can monitor the lighting levels in specific areas of your home and can send   
    alerts if the lighting levels drop below a set threshold.                                                      

Each of these detectors can be integrated into your Home I/O system, allowing you to monitor and control various   
aspects of your home's security and automation.

In [46]:
# Chain with the retriever. We need to recreate the chain because the vectorstore has changed.
# The prompt template remains the same.


# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="mmr", #  Can be "similarity" (default), "mmr", or "similarity_score_threshold".
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In [47]:
result = chain_with_retriever.invoke(input={"input": "List the available detectors in Home I/O."})

console.print(Markdown(result["answer"])) # we print the output proposed by the LLM

The available detectors in Home I/O are:                                                                           

 1 Door Detector                                                                                                   
 2 Motion Detector                                                                                                 
 3 Smoke Detector                                                                                                  

I am not sure about any other detectors, as the provided context only mentions these three types.

In [48]:
# to get an idea of the context used to answer the question we print the full results with the context.
console.print(result)

{
    'input': 'List the available detectors in Home I/O.',
    'context': [
        Document(
            id='7d4fa400-4cf8-4ecb-b9fb-4a63b052445a',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/connectio/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Using with Connect I/O - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='Using with Connect I/O Any Home I/O device set to External Mode is automatically detected
by Connect I/O. Connect I/O can be used as a SoftPLC to control directly Home I/O. Alternatively, it can be used as
an interface with external technologies, allowing Home I/O to be easily connected to PLC, microcontrollers, Modbus,
among many others technologies. Light Switch Note: Light Switch Dimmer Note: Up/Down Switch Motion Detector Smoke 
Detector Brightness Detector Note: The on/off bit is triggered'
        ),
        Document(
            id='2250a15a-2503-4028-b5fc-f26064936669',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/system-requirements/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'System Requirements - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='System Requirements'
        ),
        Document(
            id='ee6c144f-5123-411e-8750-d15ab64cb685',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/alarm/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Central Alarm and Siren - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='press 1 to arm. 2: Enter the code and press 2 to disarm. Siren Power consumption: 30 W'
        ),
        Document(
            id='8548688c-bca1-42b0-81f6-752299b749f7',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/safety-scenario/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Domestic Safety Scenario - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='Domestic Safety Scenario A scenario can only be created if it has at least one detector 
and one alarm. The first screen displays a list of existing safety scenarios. Click on Add Scenario to create a new
scenario. How this scenario controls each device type:'
        ),
        Document(
            id='28a03bc9-d6eb-4747-b966-418da816ea58',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/memory-addresses/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Memory Addresses - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='Memory Addresses Inputs Outputs Memories (read-only) Power (W)'
        ),
        Document(
            id='391eec61-ed91-41d1-b9e5-8f589a4d3d35',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/devices-map/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Devices Map - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='Devices Map Devices Map PDF Download The following maps indicate the available devices 
and their location. Lighting Lights Light Switches Roller Shades and Up/Down Switches Heaters and Thermostats 
Detectors Others'
        ),
        Document(
            id='3aaa3ca0-b5d3-4148-8139-4ddbb0f94ba6',
            metadata={
                'source': 'https://docs.realgames.co/ho

In [ ]:
# this time with the filter

# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="similarity", #  Can be "similarity" (default), "mmr", or "similarity_score_threshold".
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
        "filter": {
            'source': 'https://docs.realgames.co/homeio/en/detectors/' # Filter by source
        }
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In [54]:
result = chain_with_retriever.invoke(input={"input": "List the detectors in Home I/O."})

console.print(Markdown(result["answer"])) # we print the output proposed by the LLM

The detectors listed in the context are:                                                                           

 1 Door Detector                                                                                                   
 2 Motion Detector                                                                                                 
 3 Smoke Detector

In [55]:
# to get an idea of the context used to answer the question we print the full results with the context.
console.print(result)

{
    'input': 'List the detectors in Home I/O.',
    'context': [
        Document(
            id='98103b68-0747-4671-8c1b-81d051f16fed',
            metadata={
                'source': 'https://docs.realgames.co/homeio/en/detectors/',
                'content_type': 'text/html; charset=utf-8',
                'title': 'Detectors - Home I/O',
                'description': 'Home I/O Documentation',
                'language': 'en'
            },
            page_content='Detectors Door Detector When in Console mode, it can open the roller shades, the gates 
and turn the lights on. Motion Detector When in Console mode, it can open the roller shades, the gates and turn the
lights on if motion is detected. Smoke Detector This detector is used to detect smoke and can be triggered with a 
smoke canister. Press the 1 Key to throw a smoke canister. After being triggered, it automatically disarms in the 
absence of smoke.'
        )
    ],
    'answer': 'The detectors listed in the context are:\n\n1. Door Detector\n2. Motion Detector\n3. Smoke Detector'
}

### Conclusion Challenge 1

**TO DO**

Shortly describe here:
- The solution you implemented (which website(s) you used, which pages you extracted, which prompts you tested, etc.)
- How did you use the metadata to filter the documents? Did you get better results?
- The results you got (e.g., the answer to the question you asked, the time it took to get the answer, etc.). Was the RAG useful? Did it improve the results? Did you get better results with or without the website information?
- Limitations and possible improvements of the solution you implemented.

## Conclusion Lab - Activity 1 RAG

In this lab, we learned how to create a simple RAG system to answer questions about pdf documents and websites. We used a vector database to store information about the pdf documents and a self-hosted LLM to generate the answers. We also learned how to create prompts and interact with the model to generate text.

Pdf and web documents are just examples of the many applications of RAG systems. RAG can work with any type of document or data source, such as databases, APIs, YouTube videos, excel files, etc. 